# Cross-Language Ad Matching Research — Notebook 03
## RO4: FAISS Recommendation System, Scalability Evaluation, API, Prototype, and User Testing

### Confirmed Notebook 02 result

The selected model was **E4 — RO3 hard-negative contrastive multilingual E5**.

| Metric | Test score |
|---|---:|
| Recall@1 | 0.6290 |
| Recall@5 | 0.8510 |
| Recall@10 | 0.9070 |
| MRR@10 | 0.7243 |
| nDCG@10 | 0.7687 |

### This notebook produces

- FAISS exact and HNSW indexes
- Retrieval-quality verification
- Exact-versus-approximate comparison
- Latency and storage benchmarks
- Gradio recommendation prototype
- FastAPI deployment bundle
- Search and feedback logs
- User-testing template
- `RO4_outputs.zip`

**Before running:** Runtime → Change runtime type → T4 GPU.

## 1. Install libraries

In [ ]:
!pip -q install     sentence-transformers==5.6.0     faiss-cpu     "gradio>=6.0,<7.0"     fastapi     uvicorn     pandas     matplotlib     tqdm

## 2. Imports and reproducibility

In [ ]:
import os
import gc
import json
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import faiss
import gradio as gr

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("FAISS:", getattr(faiss, "__version__", "unknown"))
print("Gradio:", gr.__version__)
print("GPU available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable T4 GPU using Runtime → Change runtime type."
    )

print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ad_matching_research"
)
RO4_DIR = PROJECT_DIR / "ro4_artifacts"
RO4_RESULT_DIR = PROJECT_DIR / "ro4_results"

RO4_DIR.mkdir(parents=True, exist_ok=True)
RO4_RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("RO4 artifact directory:", RO4_DIR)

## 4. Upload previous outputs

Upload both files together:

1. `RO1_outputs.zip`
2. `Day2_results.zip`

In [ ]:
from google.colab import files

uploaded = files.upload()

uploaded_zip_paths = [
    Path("/content") / name
    for name in uploaded
    if name.lower().endswith(".zip")
]

RO1_ZIP = next(
    (
        path for path in uploaded_zip_paths
        if "ro1" in path.name.lower()
    ),
    None,
)
DAY2_ZIP = next(
    (
        path for path in uploaded_zip_paths
        if "day2" in path.name.lower()
    ),
    None,
)

if RO1_ZIP is None or DAY2_ZIP is None:
    raise FileNotFoundError(
        "Upload RO1_outputs.zip and Day2_results.zip."
    )

RO1_DIR = Path("/content/RO1_outputs")
DAY2_DIR = Path("/content/Day2_results")

for directory in [RO1_DIR, DAY2_DIR]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(RO1_ZIP, "r") as archive:
    archive.extractall(RO1_DIR)

with zipfile.ZipFile(DAY2_ZIP, "r") as archive:
    archive.extractall(DAY2_DIR)

print("RO1 files:", len(list(RO1_DIR.iterdir())))
print("Day 2 files:", len(list(DAY2_DIR.iterdir())))

## 5. Load data and selected model

In [ ]:
clean_df = pd.read_csv(
    RO1_DIR / "clean_pairs_with_splits.csv"
)
corpus_df = pd.read_csv(
    RO1_DIR / "offering_corpus.csv"
)
test_df = pd.read_csv(
    RO1_DIR / "test_pairs.csv"
)

with open(
    DAY2_DIR / "day2_summary.json",
    "r",
    encoding="utf-8",
) as file:
    day2_summary = json.load(file)

selected_model_name = day2_summary[
    "selected_finetuned_model"
]
selected_model_path = Path(
    day2_summary["selected_model_path"]
)

if not selected_model_path.exists():
    preferred = (
        PROJECT_DIR
        / "models"
        / "ro3_hard_negative_e5"
    )
    fallback = (
        PROJECT_DIR
        / "models"
        / "ro2_domain_finetuned_e5"
    )

    if preferred.exists():
        selected_model_path = preferred
    elif fallback.exists():
        selected_model_path = fallback
    else:
        raise FileNotFoundError(
            "The trained model is missing from Google Drive. "
            "Run Notebook 02 again."
        )

corpus_df = (
    corpus_df
    .drop_duplicates("offering_id")
    .reset_index(drop=True)
)
corpus_df["faiss_row"] = np.arange(len(corpus_df))

print("Selected model:", selected_model_name)
print("Model path:", selected_model_path)
print("Corpus size:", len(corpus_df))
print("Test queries:", len(test_df))

## 6. Enrich corpus metadata

In [ ]:
metadata_columns = [
    "offering_id",
    "offering_ad_title",
    "offering_ad_description",
]

metadata_df = (
    clean_df[
        [
            column
            for column in metadata_columns
            if column in clean_df.columns
        ]
    ]
    .drop_duplicates("offering_id")
)

corpus_df = corpus_df.merge(
    metadata_df,
    on="offering_id",
    how="left",
)

for column in [
    "offering_ad_title",
    "offering_ad_description",
]:
    if column not in corpus_df.columns:
        corpus_df[column] = ""
    corpus_df[column] = (
        corpus_df[column]
        .fillna("")
        .astype(str)
    )

empty_title = (
    corpus_df["offering_ad_title"].str.strip()
    == ""
)
corpus_df.loc[
    empty_title,
    "offering_ad_title",
] = (
    corpus_df.loc[
        empty_title,
        "offering_ad",
    ]
    .str.slice(0, 100)
)

display(
    corpus_df[
        [
            "offering_id",
            "category_1",
            "category_2",
            "offering_ad_title",
        ]
    ].head()
)

## 7. Load the selected multilingual encoder

In [ ]:
MAX_SEQ_LENGTH = 256
ENCODE_BATCH_SIZE = 128
TOP_K = 10

model = SentenceTransformer(
    str(selected_model_path)
)
model.max_seq_length = MAX_SEQ_LENGTH

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension(),
)
print(
    "Maximum sequence length:",
    model.max_seq_length,
)

## 8. Encode and persist the offering corpus

- Wanted ad prefix: `query: `
- Offering ad prefix: `passage: `
- Embeddings are normalized before inner-product search.

In [ ]:
CORPUS_EMBEDDINGS_PATH = (
    RO4_DIR / "corpus_embeddings.npy"
)
CORPUS_METADATA_PATH = (
    RO4_DIR / "corpus_metadata.csv"
)

dimension = (
    model.get_sentence_embedding_dimension()
)
rebuild_embeddings = True

if CORPUS_EMBEDDINGS_PATH.exists():
    existing = np.load(
        CORPUS_EMBEDDINGS_PATH,
        mmap_mode="r",
    )

    if (
        existing.shape[0] == len(corpus_df)
        and existing.shape[1] == dimension
    ):
        corpus_embeddings = np.asarray(
            existing,
            dtype="float32",
        )
        rebuild_embeddings = False
        print(
            "Loaded saved embeddings:",
            corpus_embeddings.shape,
        )

if rebuild_embeddings:
    passage_texts = [
        "passage: " + str(text)
        for text in corpus_df["offering_ad"]
    ]

    corpus_embeddings = model.encode(
        passage_texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    np.save(
        CORPUS_EMBEDDINGS_PATH,
        corpus_embeddings,
    )

corpus_df.to_csv(
    CORPUS_METADATA_PATH,
    index=False,
)

print("Embedding shape:", corpus_embeddings.shape)
print(
    "Mean vector norm:",
    np.linalg.norm(
        corpus_embeddings,
        axis=1,
    ).mean(),
)

# 9. Build FAISS indexes

- **IndexFlatIP:** exact reference search
- **HNSW32:** approximate scalable search

In [ ]:
FLAT_INDEX_PATH = RO4_DIR / "faiss_flat.index"
HNSW_INDEX_PATH = RO4_DIR / "faiss_hnsw.index"

dimension = corpus_embeddings.shape[1]

flat_index = faiss.IndexFlatIP(dimension)

start = time.perf_counter()
flat_index.add(corpus_embeddings)
flat_build_seconds = time.perf_counter() - start

faiss.write_index(
    flat_index,
    str(FLAT_INDEX_PATH),
)

hnsw_index = faiss.index_factory(
    dimension,
    "HNSW32,Flat",
    faiss.METRIC_INNER_PRODUCT,
)
hnsw_index.hnsw.efConstruction = 200
hnsw_index.hnsw.efSearch = 64

start = time.perf_counter()
hnsw_index.add(corpus_embeddings)
hnsw_build_seconds = time.perf_counter() - start

faiss.write_index(
    hnsw_index,
    str(HNSW_INDEX_PATH),
)

print("Flat vectors:", flat_index.ntotal)
print("HNSW vectors:", hnsw_index.ntotal)
print("Flat build seconds:", round(flat_build_seconds, 3))
print("HNSW build seconds:", round(hnsw_build_seconds, 3))
print(
    "Flat size MB:",
    round(
        FLAT_INDEX_PATH.stat().st_size
        / 1024**2,
        2,
    ),
)
print(
    "HNSW size MB:",
    round(
        HNSW_INDEX_PATH.stat().st_size
        / 1024**2,
        2,
    ),
)

# 10. Retrieval-quality verification

In [ ]:
def encode_queries(texts, batch_size=128):
    return model.encode(
        [
            "query: " + str(text)
            for text in texts
        ],
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")


def retrieval_metrics(
    retrieved_indices,
    target_indices,
):
    ranks = []

    for retrieved, target in zip(
        retrieved_indices,
        target_indices,
    ):
        positions = np.where(
            retrieved == target
        )[0]

        ranks.append(
            int(positions[0] + 1)
            if len(positions)
            else None
        )

    metrics = {}

    for k in [1, 5, 10]:
        hit_rate = np.mean(
            [
                rank is not None
                and rank <= k
                for rank in ranks
            ]
        )

        metrics[f"Recall@{k}"] = hit_rate
        metrics[f"Precision@{k}"] = (
            hit_rate / k
        )

    metrics["MRR@10"] = np.mean(
        [
            1.0 / rank
            if rank is not None
            and rank <= 10
            else 0.0
            for rank in ranks
        ]
    )

    metrics["nDCG@10"] = np.mean(
        [
            1.0 / np.log2(rank + 1)
            if rank is not None
            and rank <= 10
            else 0.0
            for rank in ranks
        ]
    )

    metrics["Top10_Failures"] = int(
        sum(rank is None for rank in ranks)
    )

    return metrics, ranks


offering_to_row = dict(
    zip(
        corpus_df["offering_id"],
        corpus_df["faiss_row"],
    )
)

target_rows = np.array(
    [
        offering_to_row[offering_id]
        for offering_id in test_df[
            "offering_id"
        ]
    ],
    dtype=np.int64,
)

test_query_embeddings = encode_queries(
    test_df["wanted_ad"].tolist()
)

flat_scores, flat_indices = flat_index.search(
    test_query_embeddings,
    TOP_K,
)

hnsw_scores, hnsw_indices = hnsw_index.search(
    test_query_embeddings,
    TOP_K,
)

flat_metrics, flat_ranks = retrieval_metrics(
    flat_indices,
    target_rows,
)

hnsw_metrics, hnsw_ranks = retrieval_metrics(
    hnsw_indices,
    target_rows,
)

quality_comparison = pd.DataFrame(
    [
        {
            "Index": "FAISS IndexFlatIP",
            **flat_metrics,
        },
        {
            "Index": "FAISS HNSW32",
            **hnsw_metrics,
        },
    ]
)

quality_comparison.to_csv(
    RO4_RESULT_DIR
    / "faiss_quality_comparison.csv",
    index=False,
)

display(quality_comparison.round(4))

## 11. Check consistency with Notebook 02

In [ ]:
reported_ro3 = day2_summary[
    "ro3_test_metrics"
]

consistency_rows = []

for metric in [
    "Recall@1",
    "Recall@5",
    "Recall@10",
    "MRR@10",
    "nDCG@10",
]:
    consistency_rows.append(
        {
            "Metric": metric,
            "Notebook_02":
                reported_ro3[metric],
            "Notebook_03_FlatIP":
                flat_metrics[metric],
            "Absolute_Difference": abs(
                reported_ro3[metric]
                - flat_metrics[metric]
            ),
        }
    )

consistency_df = pd.DataFrame(
    consistency_rows
)

consistency_df.to_csv(
    RO4_RESULT_DIR
    / "notebook_result_consistency.csv",
    index=False,
)

display(consistency_df.round(8))

## 12. HNSW approximation quality

In [ ]:
top1_agreement = np.mean(
    flat_indices[:, 0]
    == hnsw_indices[:, 0]
)

top10_overlap = np.mean(
    [
        len(
            set(exact_row).intersection(
                set(approx_row)
            )
        ) / TOP_K
        for exact_row, approx_row
        in zip(
            flat_indices,
            hnsw_indices,
        )
    ]
)

approximation_df = pd.DataFrame(
    [
        {
            "Top1_Agreement":
                top1_agreement,
            "Mean_Top10_Overlap":
                top10_overlap,
            "HNSW_efSearch":
                hnsw_index.hnsw.efSearch,
            "HNSW_efConstruction":
                hnsw_index.hnsw.efConstruction,
        }
    ]
)

approximation_df.to_csv(
    RO4_RESULT_DIR
    / "hnsw_approximation_quality.csv",
    index=False,
)

display(approximation_df.round(4))

# 13. Latency benchmark

The benchmark reports:

- Mean latency
- p50
- p95
- p99
- Sequential estimated QPS

This is not a concurrent production load test.

In [ ]:
BENCHMARK_QUERY_COUNT = min(
    500,
    len(test_df),
)
END_TO_END_QUERY_COUNT = min(
    100,
    len(test_df),
)

benchmark_queries = (
    test_df["wanted_ad"]
    .sample(
        n=BENCHMARK_QUERY_COUNT,
        random_state=SEED,
    )
    .tolist()
)

# Encoder warm-up
for query in benchmark_queries[:5]:
    _ = model.encode(
        ["query: " + query],
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

torch.cuda.synchronize()

embedding_latencies_ms = []
benchmark_embeddings = []

for query in tqdm(
    benchmark_queries,
    desc="Embedding latency",
):
    torch.cuda.synchronize()
    start = time.perf_counter()

    embedding = model.encode(
        ["query: " + query],
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    torch.cuda.synchronize()

    embedding_latencies_ms.append(
        (
            time.perf_counter() - start
        ) * 1000
    )
    benchmark_embeddings.append(
        embedding[0]
    )

benchmark_embeddings = np.asarray(
    benchmark_embeddings,
    dtype="float32",
)

# FAISS warm-up
_ = flat_index.search(
    benchmark_embeddings[:5],
    TOP_K,
)
_ = hnsw_index.search(
    benchmark_embeddings[:5],
    TOP_K,
)

flat_search_latencies_ms = []
hnsw_search_latencies_ms = []

for embedding in tqdm(
    benchmark_embeddings,
    desc="FAISS search latency",
):
    vector = embedding.reshape(1, -1)

    start = time.perf_counter()
    _ = flat_index.search(vector, TOP_K)
    flat_search_latencies_ms.append(
        (
            time.perf_counter() - start
        ) * 1000
    )

    start = time.perf_counter()
    _ = hnsw_index.search(vector, TOP_K)
    hnsw_search_latencies_ms.append(
        (
            time.perf_counter() - start
        ) * 1000
    )

end_to_end_latencies_ms = []

for query in tqdm(
    benchmark_queries[
        :END_TO_END_QUERY_COUNT
    ],
    desc="End-to-end latency",
):
    torch.cuda.synchronize()
    start = time.perf_counter()

    vector = model.encode(
        ["query: " + query],
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    _ = hnsw_index.search(
        vector,
        TOP_K,
    )

    torch.cuda.synchronize()

    end_to_end_latencies_ms.append(
        (
            time.perf_counter() - start
        ) * 1000
    )


def latency_summary(component, values):
    values = np.asarray(
        values,
        dtype=float,
    )

    return {
        "Component": component,
        "Queries": len(values),
        "Mean_ms": values.mean(),
        "P50_ms": np.percentile(
            values,
            50,
        ),
        "P95_ms": np.percentile(
            values,
            95,
        ),
        "P99_ms": np.percentile(
            values,
            99,
        ),
        "Sequential_Estimated_QPS":
            1000.0 / values.mean(),
    }


latency_df = pd.DataFrame(
    [
        latency_summary(
            "Query embedding on Colab GPU",
            embedding_latencies_ms,
        ),
        latency_summary(
            "FAISS FlatIP search on CPU",
            flat_search_latencies_ms,
        ),
        latency_summary(
            "FAISS HNSW search on CPU",
            hnsw_search_latencies_ms,
        ),
        latency_summary(
            "End-to-end embedding + HNSW",
            end_to_end_latencies_ms,
        ),
    ]
)

latency_df.to_csv(
    RO4_RESULT_DIR
    / "latency_benchmark.csv",
    index=False,
)

display(latency_df.round(4))

## 14. Storage and build-time summary

In [ ]:
storage_df = pd.DataFrame(
    [
        {
            "Artifact":
                "Corpus embeddings",
            "Size_MB":
                CORPUS_EMBEDDINGS_PATH.stat().st_size
                / 1024**2,
            "Build_Seconds": np.nan,
        },
        {
            "Artifact":
                "FAISS FlatIP index",
            "Size_MB":
                FLAT_INDEX_PATH.stat().st_size
                / 1024**2,
            "Build_Seconds":
                flat_build_seconds,
        },
        {
            "Artifact":
                "FAISS HNSW32 index",
            "Size_MB":
                HNSW_INDEX_PATH.stat().st_size
                / 1024**2,
            "Build_Seconds":
                hnsw_build_seconds,
        },
        {
            "Artifact":
                "Corpus metadata CSV",
            "Size_MB":
                CORPUS_METADATA_PATH.stat().st_size
                / 1024**2,
            "Build_Seconds": np.nan,
        },
    ]
)

storage_df.to_csv(
    RO4_RESULT_DIR
    / "artifact_storage_summary.csv",
    index=False,
)

display(storage_df.round(3))

# 15. Recommendation and feedback functions

In [ ]:
SEARCH_LOG_PATH = (
    RO4_RESULT_DIR
    / "prototype_search_log.csv"
)
FEEDBACK_LOG_PATH = (
    RO4_RESULT_DIR
    / "prototype_feedback_log.csv"
)

CATEGORY_OPTIONS = [
    "All",
] + sorted(
    corpus_df["category_1"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


def recommend_ads(
    query,
    category="All",
    top_k=5,
):
    query = str(query or "").strip()

    empty_output = pd.DataFrame(
        columns=[
            "Rank",
            "Score",
            "Category",
            "Subcategory",
            "Title",
            "Advertisement",
        ]
    )

    if not query:
        return (
            empty_output,
            "Please enter a wanted advertisement.",
        )

    top_k = int(top_k)
    start = time.perf_counter()

    query_embedding = model.encode(
        ["query: " + query],
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    search_k = (
        top_k
        if category == "All"
        else min(
            max(top_k * 40, 500),
            len(corpus_df),
        )
    )

    scores, indices = hnsw_index.search(
        query_embedding,
        search_k,
    )

    rows = []

    for score, corpus_row in zip(
        scores[0],
        indices[0],
    ):
        if corpus_row < 0:
            continue

        item = corpus_df.iloc[
            int(corpus_row)
        ]

        if (
            category != "All"
            and str(item["category_1"])
            != category
        ):
            continue

        rows.append(
            {
                "Rank": len(rows) + 1,
                "Score": round(
                    float(score),
                    4,
                ),
                "Category":
                    item["category_1"],
                "Subcategory":
                    item["category_2"],
                "Title":
                    item[
                        "offering_ad_title"
                    ],
                "Advertisement":
                    item["offering_ad"],
            }
        )

        if len(rows) >= top_k:
            break

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000

    output_df = pd.DataFrame(rows)

    pd.DataFrame(
        [
            {
                "timestamp_utc":
                    pd.Timestamp.utcnow(),
                "query": query,
                "category_filter":
                    category,
                "top_k": top_k,
                "returned_results":
                    len(output_df),
                "latency_ms":
                    elapsed_ms,
            }
        ]
    ).to_csv(
        SEARCH_LOG_PATH,
        mode="a",
        header=not SEARCH_LOG_PATH.exists(),
        index=False,
    )

    status = (
        f"Returned {len(output_df)} recommendations "
        f"in {elapsed_ms:.2f} ms."
    )

    return output_df, status


def save_feedback(
    query,
    rating,
    relevant,
    comments,
):
    query = str(query or "").strip()

    if not query:
        return (
            "Enter the tested query before "
            "submitting feedback."
        )

    pd.DataFrame(
        [
            {
                "timestamp_utc":
                    pd.Timestamp.utcnow(),
                "query": query,
                "overall_rating_1_to_5":
                    rating,
                "recommendations_relevant":
                    relevant,
                "comments":
                    str(comments or "").strip(),
            }
        ]
    ).to_csv(
        FEEDBACK_LOG_PATH,
        mode="a",
        header=not FEEDBACK_LOG_PATH.exists(),
        index=False,
    )

    return "Feedback saved successfully."


sample_query = test_df.iloc[0]["wanted_ad"]
sample_results, sample_status = recommend_ads(
    sample_query,
    "All",
    5,
)

print(sample_status)
display(sample_results)

# 16. Create a user-testing pack

Use 5–10 participants where possible. Report this as a small prototype usability evaluation, not a representative population study.

In [ ]:
USER_TEST_QUERY_COUNT = 20

sample_cases = (
    test_df.sample(
        n=min(
            USER_TEST_QUERY_COUNT,
            len(test_df),
        ),
        random_state=SEED,
    )[
        [
            "wanted_ad",
            "category_1",
            "category_2",
            "wanted_script_group",
        ]
    ]
    .reset_index(drop=True)
)

rows = []

for case_id, row in tqdm(
    sample_cases.iterrows(),
    total=len(sample_cases),
    desc="User-test cases",
):
    recommendations, _ = recommend_ads(
        row["wanted_ad"],
        "All",
        5,
    )

    titles = (
        recommendations["Title"].tolist()
        if not recommendations.empty
        else []
    )

    rows.append(
        {
            "case_id": case_id + 1,
            "participant_id": "",
            "wanted_query":
                row["wanted_ad"],
            "expected_main_category":
                row["category_1"],
            "expected_subcategory":
                row["category_2"],
            "script_group":
                row["wanted_script_group"],
            "recommended_title_1":
                titles[0]
                if len(titles) > 0
                else "",
            "recommended_title_2":
                titles[1]
                if len(titles) > 1
                else "",
            "recommended_title_3":
                titles[2]
                if len(titles) > 2
                else "",
            "recommended_title_4":
                titles[3]
                if len(titles) > 3
                else "",
            "recommended_title_5":
                titles[4]
                if len(titles) > 4
                else "",
            "top_5_relevance_1_to_5": "",
            "at_least_one_useful_yes_no": "",
            "system_easy_to_use_1_to_5": "",
            "response_speed_1_to_5": "",
            "comments": "",
        }
    )

user_testing_template = pd.DataFrame(
    rows
)

user_testing_template.to_csv(
    RO4_RESULT_DIR
    / "user_testing_template.csv",
    index=False,
)

questionnaire = '''
# Prototype User-Testing Questionnaire

Rate each item from 1 to 5.

1. The recommended advertisements matched the meaning of my query.
2. At least one of the top five recommendations was useful.
3. The system was easy to use.
4. The response speed was acceptable.
5. Sinhala, English, or code-mixed text was understood correctly.

Open question:

What was the main problem, if any, with the recommendations?

Report:
- Number of participants
- Mean relevance score
- Percentage finding at least one useful result
- Mean ease-of-use score
- Mean response-speed score
- Common comments
'''.strip()

(
    RO4_RESULT_DIR
    / "user_testing_questionnaire.md"
).write_text(
    questionnaire,
    encoding="utf-8",
)

display(user_testing_template.head(3))

# 17. Generate a portable FastAPI bundle

In [ ]:
BUNDLE_DIR = Path(
    "/content/ad_matching_prototype_bundle"
)

if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

BUNDLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    HNSW_INDEX_PATH,
    BUNDLE_DIR / "faiss_hnsw.index",
)
shutil.copy2(
    CORPUS_METADATA_PATH,
    BUNDLE_DIR / "corpus_metadata.csv",
)

system_config = {
    "model_folder":
        "models/ro3_hard_negative_e5",
    "max_seq_length": MAX_SEQ_LENGTH,
    "index_type": "HNSW32",
    "metric":
        "inner_product_on_normalized_vectors",
    "query_prefix": "query: ",
    "passage_prefix": "passage: ",
    "default_top_k": 5,
    "corpus_size": int(len(corpus_df)),
    "selected_model":
        selected_model_name,
}

with open(
    BUNDLE_DIR / "system_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        system_config,
        file,
        indent=2,
        ensure_ascii=False,
    )

(BUNDLE_DIR / "requirements.txt").write_text(
    'sentence-transformers==5.6.0\nfaiss-cpu\nfastapi\nuvicorn[standard]\npandas\nnumpy\npydantic',
    encoding="utf-8",
)

(BUNDLE_DIR / "app.py").write_text(
    'import json\nimport os\nimport time\nfrom pathlib import Path\nfrom typing import Optional\n\nimport faiss\nimport pandas as pd\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nfrom sentence_transformers import SentenceTransformer\n\nBASE_DIR = Path(__file__).resolve().parent\n\nwith open(\n    BASE_DIR / "system_config.json",\n    "r",\n    encoding="utf-8",\n) as file:\n    CONFIG = json.load(file)\n\nMODEL_PATH = Path(\n    os.getenv(\n        "MODEL_PATH",\n        BASE_DIR / CONFIG["model_folder"],\n    )\n)\n\nif not MODEL_PATH.exists():\n    raise FileNotFoundError(\n        f"Model folder not found: {MODEL_PATH}. "\n        "Copy the trained model into the models folder "\n        "or set the MODEL_PATH environment variable."\n    )\n\nMODEL = SentenceTransformer(str(MODEL_PATH))\nMODEL.max_seq_length = CONFIG["max_seq_length"]\n\nINDEX = faiss.read_index(\n    str(BASE_DIR / "faiss_hnsw.index")\n)\n\nCORPUS = pd.read_csv(\n    BASE_DIR / "corpus_metadata.csv"\n)\n\napp = FastAPI(\n    title="Sri Lankan Code-Mixed Ad Recommender",\n    version="1.0.0",\n)\n\n\nclass RecommendationRequest(BaseModel):\n    query: str = Field(min_length=1)\n    top_k: int = Field(default=5, ge=1, le=20)\n    category: Optional[str] = None\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "ok",\n        "corpus_size": int(INDEX.ntotal),\n        "model": CONFIG["selected_model"],\n    }\n\n\n@app.post("/recommend")\ndef recommend(request: RecommendationRequest):\n    query = request.query.strip()\n\n    if not query:\n        raise HTTPException(\n            status_code=400,\n            detail="Query cannot be empty.",\n        )\n\n    start = time.perf_counter()\n\n    vector = MODEL.encode(\n        ["query: " + query],\n        convert_to_numpy=True,\n        normalize_embeddings=True,\n        show_progress_bar=False,\n    ).astype("float32")\n\n    search_k = (\n        request.top_k\n        if not request.category\n        else min(\n            max(request.top_k * 40, 500),\n            len(CORPUS),\n        )\n    )\n\n    scores, indices = INDEX.search(\n        vector,\n        search_k,\n    )\n\n    recommendations = []\n\n    for score, row_index in zip(\n        scores[0],\n        indices[0],\n    ):\n        if row_index < 0:\n            continue\n\n        item = CORPUS.iloc[int(row_index)]\n\n        if (\n            request.category\n            and str(item["category_1"])\n            != request.category\n        ):\n            continue\n\n        recommendations.append(\n            {\n                "rank": len(recommendations) + 1,\n                "score": round(float(score), 4),\n                "offering_id": str(item["offering_id"]),\n                "category": str(item["category_1"]),\n                "subcategory": str(item["category_2"]),\n                "title": str(item["offering_ad_title"]),\n                "advertisement": str(item["offering_ad"]),\n            }\n        )\n\n        if len(recommendations) >= request.top_k:\n            break\n\n    latency_ms = (\n        time.perf_counter() - start\n    ) * 1000\n\n    return {\n        "query": query,\n        "latency_ms": round(latency_ms, 2),\n        "recommendations": recommendations,\n    }',
    encoding="utf-8",
)

(BUNDLE_DIR / "README.md").write_text(
    '# Sri Lankan Code-Mixed Ad Recommendation API\n\n## Add the trained model\n\nCopy the trained model folder to:\n\nmodels/ro3_hard_negative_e5\n\n## Install\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate\n# Windows: .venv\\Scripts\\activate\npip install -r requirements.txt\n```\n\n## Run\n\n```bash\nuvicorn app:app --host 0.0.0.0 --port 8000\n```\n\nHealth endpoint:\n\nhttp://localhost:8000/health\n\nInteractive API documentation:\n\nhttp://localhost:8000/docs',
    encoding="utf-8",
)

print("Bundle files:")
for path in sorted(BUNDLE_DIR.iterdir()):
    print("-", path.name)

# 18. Save final summary and download `RO4_outputs.zip`

In [ ]:
deployment_choice = (
    "FAISS HNSW32"
    if (
        top10_overlap >= 0.95
        and hnsw_metrics["Recall@10"]
        >= flat_metrics["Recall@10"]
        - 0.01
    )
    else "FAISS IndexFlatIP"
)

ro4_summary = {
    "selected_model":
        selected_model_name,
    "selected_model_path":
        str(selected_model_path),
    "corpus_size":
        int(len(corpus_df)),
    "embedding_dimension":
        int(corpus_embeddings.shape[1]),
    "flat_metrics": {
        key: (
            int(value)
            if key == "Top10_Failures"
            else float(value)
        )
        for key, value
        in flat_metrics.items()
    },
    "hnsw_metrics": {
        key: (
            int(value)
            if key == "Top10_Failures"
            else float(value)
        )
        for key, value
        in hnsw_metrics.items()
    },
    "hnsw_top1_agreement":
        float(top1_agreement),
    "hnsw_mean_top10_overlap":
        float(top10_overlap),
    "recommended_deployment_index":
        deployment_choice,
}

with open(
    RO4_RESULT_DIR / "ro4_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        ro4_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

pd.DataFrame(
    [
        {
            "Component": "Text encoder",
            "Implementation":
                selected_model_name,
        },
        {
            "Component": "Vector database",
            "Implementation":
                deployment_choice,
        },
        {
            "Component": "Corpus",
            "Implementation":
                f"{len(corpus_df):,} unique offering ads",
        },
        {
            "Component": "Input",
            "Implementation":
                "English, Sinhala, Singlish, or code-mixed wanted ad",
        },
        {
            "Component": "Output",
            "Implementation":
                "Ranked Top-K offering ads",
        },
    ]
).to_csv(
    RO4_RESULT_DIR
    / "final_system_components.csv",
    index=False,
)

shutil.make_archive(
    str(
        RO4_RESULT_DIR
        / "ad_matching_prototype_bundle"
    ),
    "zip",
    BUNDLE_DIR,
)

for source in [
    FLAT_INDEX_PATH,
    HNSW_INDEX_PATH,
    CORPUS_METADATA_PATH,
]:
    shutil.copy2(
        source,
        RO4_RESULT_DIR / source.name,
    )

ro4_outputs_zip = shutil.make_archive(
    "/content/RO4_outputs",
    "zip",
    RO4_RESULT_DIR,
)

print(
    json.dumps(
        ro4_summary,
        indent=2,
        ensure_ascii=False,
    )
)
print("Created:", ro4_outputs_zip)

from google.colab import files
files.download(ro4_outputs_zip)

# Thesis-ready RO4 methodology

> The selected hard-negative contrastive multilingual encoder was integrated with FAISS for dense retrieval. Each unique offering advertisement was encoded once using the passage prefix and stored as a normalized dense vector. Wanted advertisements were encoded online using the query prefix. IndexFlatIP was evaluated as the exact-search reference, while HNSW32 was evaluated as an approximate deployment option. Because all embeddings were normalized, inner-product ranking corresponded to cosine-similarity ranking. Retrieval quality was verified on the held-out test set using Recall@K, MRR@10, and nDCG@10. HNSW was compared with exact search using top-one agreement, Top-10 overlap, and ground-truth retrieval metrics. Scalability was evaluated through index size, build time, query-embedding latency, vector-search latency, end-to-end latency, and sequential estimated throughput. The model was exposed through a Gradio prototype and a FastAPI endpoint, with search and feedback logging for prototype user evaluation.

## Reporting warning

Do not claim 10,000 queries per second unless the measured benchmark proves it. State clearly that the Colab benchmark uses GPU encoding, CPU FAISS search, and sequential requests.

# 19. Launch the Gradio prototype

Run this final cell manually after downloading the ZIP. The public share link is temporary.

In [ ]:
with gr.Blocks(
    title="Sri Lankan Code-Mixed Ad Recommender"
) as demo:
    gr.Markdown(
        '''
        # Sri Lankan Code-Mixed Ad Recommender

        Enter a wanted advertisement in English,
        Sinhala, Singlish, or a code-mixed form.
        '''
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Wanted advertisement",
                lines=4,
                placeholder=(
                    "Samsung phone ekak ganna one"
                ),
            )

            category_input = gr.Dropdown(
                choices=CATEGORY_OPTIONS,
                value="All",
                label="Optional category filter",
            )

            top_k_input = gr.Slider(
                minimum=1,
                maximum=10,
                value=5,
                step=1,
                label="Number of recommendations",
            )

            search_button = gr.Button(
                "Find Matching Ads",
                variant="primary",
            )

        with gr.Column(scale=3):
            result_table = gr.Dataframe(
                headers=[
                    "Rank",
                    "Score",
                    "Category",
                    "Subcategory",
                    "Title",
                    "Advertisement",
                ],
                interactive=False,
                wrap=True,
                label="Recommendations",
            )
            status_output = gr.Markdown()

    gr.Markdown("## Prototype Feedback")

    with gr.Row():
        rating_input = gr.Slider(
            minimum=1,
            maximum=5,
            value=3,
            step=1,
            label="Overall quality",
        )

        relevant_input = gr.Radio(
            choices=[
                "Yes",
                "No",
                "Partly",
            ],
            label=(
                "Was at least one result useful?"
            ),
        )

    comments_input = gr.Textbox(
        label="Comments",
        lines=2,
    )

    feedback_button = gr.Button(
        "Save Feedback"
    )
    feedback_status = gr.Markdown()

    search_button.click(
        fn=recommend_ads,
        inputs=[
            query_input,
            category_input,
            top_k_input,
        ],
        outputs=[
            result_table,
            status_output,
        ],
    )

    feedback_button.click(
        fn=save_feedback,
        inputs=[
            query_input,
            rating_input,
            relevant_input,
            comments_input,
        ],
        outputs=feedback_status,
    )

    gr.Examples(
        examples=[
            [
                "Samsung phone ekak "
                "ganna one"
            ],
            [
                "ගාල්ලෙන් ඉඩමක් අවශ්‍යයි"
            ],
            [
                "Toyota car ekak hoyanawa"
            ],
            [
                "used laptop එකක් ඕන"
            ],
        ],
        inputs=query_input,
    )

demo.launch(
    share=True,
    debug=False,
)